In [1]:
from urllib.parse import quote_plus
import numpy as np
import pandas as pd
from scipy import stats
from sqlalchemy import create_engine
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
from getpass import getpass
import os
from dotenv import load_dotenv


load_dotenv()  

DB_USER = os.getenv("DB_USER", "root")
DB_PASS = os.getenv("DB_PASS")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_NAME = os.getenv("DB_NAME", "olist_dw")

engine = create_engine(f"mysql+pymysql://{DB_USER}:{quote_plus(DB_PASS)}@{DB_HOST}/{DB_NAME}")

print("=" * 70)
print("OLIST STATISTICAL A/B TESTING & HYPOTHESIS SUITE")
print("=" * 70)
print(pd.read_sql("SELECT COUNT(*) AS mql_count FROM vw_marketing_funnel", engine))
# ==============================================================================
# EXPERIMENT 1: B2B Marketing Funnel Channel Conversion (Paid vs. Organic Search)
# ==============================================================================
df_funnel = pd.read_sql("""
    SELECT acquisition_channel, is_closed_won
    FROM vw_marketing_funnel
    WHERE acquisition_channel IN ('paid_search', 'organic_search')
""", engine)

summary_funnel = df_funnel.groupby("acquisition_channel")["is_closed_won"].agg(
    leads="count", conversions="sum", win_rate="mean"
)
print("\n[EXPERIMENT 1] B2B Funnel Conversion: Paid Search vs. Organic Search")
print(summary_funnel)

counts = np.array([
    summary_funnel.loc["paid_search", "conversions"],
    summary_funnel.loc["organic_search", "conversions"]
])
nobs = np.array([
    summary_funnel.loc["paid_search", "leads"],
    summary_funnel.loc["organic_search", "leads"]
])

z_stat, p_val_z = proportions_ztest(counts, nobs, alternative="two-sided")
ci_low, ci_high = proportion_confint(counts, nobs, alpha=0.05, method="normal")
rel_lift = ((summary_funnel.loc["paid_search", "win_rate"] -
             summary_funnel.loc["organic_search", "win_rate"]) /
            summary_funnel.loc["organic_search", "win_rate"]) * 100

print(f"-> Two-Proportion Z-Stat: {z_stat:.4f} | p-value: {p_val_z:.4f}")
print(f"-> Relative Lift (Paid vs Organic): {rel_lift:+.2f}%")
print(f"-> 95% CI Paid Search: [{ci_low[0]:.2%}, {ci_high[0]:.2%}] | Organic: [{ci_low[1]:.2%}, {ci_high[1]:.2%}]")
print("-> Conclusion:", "STATISTICALLY SIGNIFICANT difference (Reject H0)" if p_val_z < 0.05 else "NO significant difference (Fail to reject H0)")

# ==============================================================================
# EXPERIMENT 2: Logistics SLA Impact on Review Score (On-Time vs. Late)
# ==============================================================================
df_sla = pd.read_sql("""
    SELECT sla_status, review_score
    FROM vw_fact_orders
    WHERE order_status = 'delivered'
      AND sla_status IN ('On-Time', 'Late')
      AND review_score IS NOT NULL
""", engine)

ontime_scores = df_sla[df_sla["sla_status"] == "On-Time"]["review_score"]
late_scores = df_sla[df_sla["sla_status"] == "Late"]["review_score"]

t_stat, p_val_t = stats.ttest_ind(ontime_scores, late_scores, equal_var=False)
u_stat, p_val_u = stats.mannwhitneyu(ontime_scores, late_scores, alternative="two-sided")

print("\n" + "-" * 70)
print("[EXPERIMENT 2] Delivery SLA Impact on Customer Review Score")
print(f"-> On-Time Mean: {ontime_scores.mean():.2f} (n={len(ontime_scores):,}) | Late Mean: {late_scores.mean():.2f} (n={len(late_scores):,})")
print(f"-> Mean Rating Drop from Late Delivery: {late_scores.mean() - ontime_scores.mean():.2f} stars")
print(f"-> Welch's T-Stat: {t_stat:.4f} | p-value: {p_val_t:.4e}")
print(f"-> Mann-Whitney U p-value (Non-parametric check): {p_val_u:.4e}")

# ==============================================================================
# EXPERIMENT 3: Power Analysis for Loyalty Voucher A/B Test Design
# ==============================================================================
baseline_rate = 0.030   # Current 3.0% repeat purchase rate
target_rate = 0.042     # Target 4.2% (+40% relative lift)
effect_size = proportion_effectsize(target_rate, baseline_rate)
power_analysis = NormalIndPower()
sample_size = power_analysis.solve_power(effect_size=effect_size, power=0.80, alpha=0.05, ratio=1.0)

print("\n" + "-" * 70)
print("[EXPERIMENT 3] A/B Test Power Analysis (Post-Delivery Loyalty Voucher)")
print(f"-> Baseline Repeat Rate: {baseline_rate:.1%} | Target MDE Rate: {target_rate:.1%}")
print(f"-> Required Sample Size per Variant (80% Power, alpha=0.05): {int(np.ceil(sample_size)):,} customers")
print("=" * 70)

OLIST STATISTICAL A/B TESTING & HYPOTHESIS SUITE
   mql_count
0       8000

[EXPERIMENT 1] B2B Funnel Conversion: Paid Search vs. Organic Search
                     leads  conversions  win_rate
acquisition_channel                              
organic_search        2296          271  0.118031
paid_search           1586          195  0.122951
-> Two-Proportion Z-Stat: 0.4636 | p-value: 0.6429
-> Relative Lift (Paid vs Organic): +4.17%
-> 95% CI Paid Search: [10.68%, 13.91%] | Organic: [10.48%, 13.12%]
-> Conclusion: NO significant difference (Fail to reject H0)

----------------------------------------------------------------------
[EXPERIMENT 2] Delivery SLA Impact on Customer Review Score
-> On-Time Mean: 4.29 (n=89,443) | Late Mean: 2.27 (n=6,381)
-> Mean Rating Drop from Late Delivery: -2.02 stars
-> Welch's T-Stat: 100.7444 | p-value: 0.0000e+00
-> Mann-Whitney U p-value (Non-parametric check): 0.0000e+00

----------------------------------------------------------------------
[EXP